# Phase 1: Project Planning

## a) Problem Statement

**Problem:**  
Toronto’s mental‑health outcomes (e.g. coping scores, life‑stress percentage) are reported at the level of seven broad clusters, while green‑space data and neighbourhood census profiles exist at much finer spatial resolution. This misalignment makes it difficult to assess whether access to green spaces—parks, open areas, and other amenities—relates to mental‑health outcomes.

**Value of Solving:**  
By integrating these data silos into a unified cluster‑level framework, we can:
- Quantify the relationship between green‑space access and self‑reported mental‑health metrics.  
- Identify clusters with both poor green‑space access and adverse mental‑health indicators.  
- Inform targeted interventions by public‑health teams and urban planners to improve equity in mental well‑being.

## b) Data Sources

1. **Mental‑Health Indicators**  
   - **Origin:** Public‑health “Mental health and substance use” Excel download  
   - **Contents:** Adult self‑report indicators (Coping %, Life‑stress %, Life satisfaction %, Happiness %, Sense of belonging %) stratified by “Sub‑Toronto Geography Clusters”  
   - **Challenges:** Inconsistent naming conventions; only cluster‑level metrics (no neighbourhood breakdown)

2. **Green‑Space Inventory**  
   - **Origin:** City of Toronto “Green Spaces.csv”  
   - **Contents:** Polygon geometries of all parks and green features, with detailed `AREA_CLASS` labels  
   - **Challenges:** Geometry stored as JSON strings; heterogeneous amenity types requiring quality weighting

3. **Neighbourhood Census Profiles**  
   - **Origin:** “neighbourhood‑profiles‑2021.xlsx”  
   - **Contents:** Demographic and socioeconomic variables for 140 Toronto neighbourhoods (age groups, income, household size, immigration, education, visible‑minority counts, etc.)  
   - **Challenges:** Wide table with neighbourhoods as columns; requires transpose and cleaning; occasional mismatches with lookup keys

4. **Cluster Lookup**  
   - **Origin:** Manually created Excel template (`nh_to_cluster.xlsx`) mapping each of the 140 neighbourhoods to one of seven mental‑health clusters  
   - **Contents:** Two columns: `AREA_NAME` (neighbourhood) and `ClusterName`

# Phase 2: Project Setup

This section initializes our environment and prepares data for analysis.

In [ ]:
#a) Environment Setup

# Import core libraries
import os
import json
import pandas as pd
import geopandas as gpd
import numpy as np

# Spatial and plotting utilities
import matplotlib.pyplot as plt

# Statistical modeling
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Spatial analysis
import libpysal
from esda.moran import Moran
from spreg import ML_Lag

# Scikit-learn 
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# b) Define paths & constants

BASE_DIR      = os.path.expanduser("~/Final-Project")
DATA_DIR      = os.path.join(BASE_DIR, "data")
SHAPEFILE     = os.path.join(DATA_DIR, "Neighbourhoods - 4326", "Neighbourhoods - 4326.shp")
GREEN_CSV     = os.path.join(DATA_DIR, "Green Spaces.csv")
MENTAL_XLS    = os.path.join(DATA_DIR, "Mental health and substance use download.xlsx")
PROFILES_XLS  = os.path.join(DATA_DIR, "neighbourhood-profiles-2021.xlsx")
LOOKUP_XLSX   = os.path.join(DATA_DIR, "nh_to_cluster.xlsx")

# Target coordinate reference system for spatial joins
TARGET_CRS    = "EPSG:4326"
PROJECTED_CRS = "EPSG:3347"  # NAD83 / Statistics Canada Lambert


NameError: name 'os' is not defined

In [ ]:
# Verify that all required files exist
for path in [SHAPEFILE, GREEN_CSV, MENTAL_XLS, PROFILES_XLS, LOOKUP_XLSX]:
    assert os.path.exists(path), f"Missing: {path}"
print("All data files are present.")

 ### Data Loading & Initial Cleaning

- Read and standardize spatial layers
- Parse green-space geometries
- Read mental-health and census tables
- Load neighbourhood-to-cluster lookup


In [ ]:
# Helper: parse geometry field (JSON string or dict)
def parse_geometry(entry):
    if isinstance(entry, str):
        return shape(json.loads(entry))
    if isinstance(entry, dict):
        return shape(entry)
    return entry


In [ ]:
# 1) Load neighbourhood polygons
neigh_gdf = (
    gpd.read_file(SHAPEFILE)
       .rename(columns={"AREA_NA7": "Neighbourhood"})
       .to_crs(TARGET_CRS)
)
print(f"Loaded {len(neigh_gdf)} neighbourhood polygons.")

In [ ]:
# 2) Load green spaces, parse geometries
green_df = pd.read_csv(GREEN_CSV)
green_df['geometry'] = green_df['geometry'].apply(parse_geometry)
green_gdf = gpd.GeoDataFrame(
    green_df,
    geometry='geometry',
    crs=TARGET_CRS
)
print(f"Loaded {len(green_gdf)} green-space features.")

In [ ]:
# 3) Read mental-health indicators
mh_df = pd.read_excel(MENTAL_XLS, sheet_name=0)
print(f"Loaded {mh_df.shape[0]} mental-health records.")

In [ ]:
# 4) Read neighbourhood census profiles
profiles_df = pd.read_excel(PROFILES_XLS, sheet_name='hd2021_census_profile')
print(f"Loaded census table with shape {profiles_df.shape}.")


In [ ]:
# 5) Read neighbourhood -> cluster lookup
lookup_df = pd.read_excel(LOOKUP_XLSX)
lookup_df.columns = ['Neighbourhood', 'ClusterName']
print(f"Loaded lookup for {lookup_df.ClusterName.nunique()} clusters.")
